## 1. Setup and load the held-out test set

The notebook searches the current folder and its parents for the project data directory. It also checks the Google Drive training folder used by this project. If the test file is not found, make sure Google Drive is mounted (in Colab) and Notebook 01 has saved `processed_data/test.json` or `data/processed/test.json`.

In [1]:
import json
import re
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd

LABELS = [
    "PERSON",
    "ADDRESS",
    "PHONE",
    "AADHAAR",
    "PAN",
    "EMAIL",
    "UPI",
]

def locate_processed_dir():
    candidates = []
    cwd = Path.cwd()
    candidates.extend([cwd, *cwd.parents])

    # Project-specific Google Drive locations used in this workflow.
    candidates.extend([
        Path("/content/drive/MyDrive/IndiaPII_Training./processed_data"),
        Path("/content/drive/MyDrive/IndiaPII_Training/processed_data"),
        Path("/content/drive/MyDrive/IndiaPII_Training./data/processed"),
        Path("/content/drive/MyDrive/IndiaPII_Training/data/processed"),
    ])

    seen = set()
    for base in candidates:
        for candidate in (base / "data" / "processed", base / "processed_data", base):
            key = str(candidate.resolve()) if candidate.exists() else str(candidate)
            if key in seen:
                continue
            seen.add(key)
            if (candidate / "test.json").is_file():
                return candidate

    raise FileNotFoundError(
        "Could not find test.json. Mount Google Drive in Colab if needed, "
        "then confirm Notebook 01 saved the processed test split."
    )

PROCESSED_DIR = locate_processed_dir()
PROJECT_ROOT = PROCESSED_DIR.parent.parent if PROCESSED_DIR.name == "processed" else PROCESSED_DIR.parent
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

with open(PROCESSED_DIR / "test.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

assert test_data, "The test dataset is empty."
assert all(key in test_data[0] for key in ["text", "tokens", "ner_tags", "language"]), (
    "Unexpected test.json schema. Expected text, tokens, ner_tags, and language."
)

print("Processed directory:", PROCESSED_DIR)
print("Results directory:", RESULTS_DIR)
print("Test samples:", len(test_data))
print("Languages:", dict(Counter(x["language"] for x in test_data)))

Processed directory: /Users/navyaa09/IndiaPII/data/processed
Results directory: /Users/navyaa09/IndiaPII/results
Test samples: 140
Languages: {'hinglish': 40, 'hi': 60, 'en': 40}


## 2. Regex baseline

The Regex baseline detects structured patterns. It intentionally does not attempt PERSON recognition because a general-purpose name detector cannot be built reliably from a simple regular expression. The address pattern is English-oriented.

In [2]:
REGEX_PATTERNS = {
    "PHONE": re.compile(
        r"(?<!\d)(?:\+91[\s-]?)?[6-9]\d{9}(?!\d)"
    ),
    "AADHAAR": re.compile(
        r"(?<!\d)(?:\d{4}[\s-]?\d{4}[\s-]?\d{4})(?!\d)"
    ),
    "PAN": re.compile(
        r"(?i)(?<![A-Z])[A-Z]{5}\d{4}[A-Z](?![A-Z])"
    ),
    "EMAIL": re.compile(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
    ),
    "UPI": re.compile(
        r"(?i)\b[A-Za-z0-9._-]+@(?:okaxis|ybl|oksbi|paytm|okhdfcbank|ibl)\b"
    ),
    "ADDRESS": re.compile(
        r"(?i)\b(?:Flat|Sector|Plot|House|Road|Street|Layout|Nagar|Colony)\s+[A-Za-z0-9., -]{2,50}"
    ),
}

def regex_detect(text):
    predictions = []
    for label, pattern in REGEX_PATTERNS.items():
        for match in pattern.finditer(text):
            predictions.append({
                "start": match.start(),
                "end": match.end(),
                "label": label,
                "text": match.group(),
            })
    predictions.sort(key=lambda x: (x["start"], x["end"], x["label"]))
    return predictions

sample_text = (
    "Rahul Sharma's phone is 9000000001 and his PAN is ABCDE1234F. "
    "Email: rahul.test@example.com; UPI: rahul@okaxis; Aadhaar: 0000 0000 0001."
)
print("Sample detections:")
for item in regex_detect(sample_text):
    print(item)

Sample detections:
{'start': 24, 'end': 34, 'label': 'PHONE', 'text': '9000000001'}
{'start': 50, 'end': 60, 'label': 'PAN', 'text': 'ABCDE1234F'}
{'start': 69, 'end': 91, 'label': 'EMAIL', 'text': 'rahul.test@example.com'}
{'start': 98, 'end': 110, 'label': 'UPI', 'text': 'rahul@okaxis'}
{'start': 121, 'end': 135, 'label': 'AADHAAR', 'text': '0000 0000 0001'}


## 3. Convert BIO annotations to exact character spans

Notebook 01 stores tokens and BIO tags. This function reconstructs character offsets by locating each token in the original text in sequence. It checks alignment and tag consistency so that incorrect offsets do not silently contaminate evaluation.

In [3]:
def bio_to_entities(sample):
    text = sample["text"]
    tokens = sample["tokens"]
    tags = sample["ner_tags"]

    if len(tokens) != len(tags):
        raise ValueError(
            f"Token/tag length mismatch: {len(tokens)} tokens vs {len(tags)} tags"
        )

    entities = []
    search_position = 0
    current_entity = None

    def close_entity():
        nonlocal current_entity
        if current_entity is not None:
            entities.append(current_entity)
            current_entity = None

    for token, tag in zip(tokens, tags):
        if isinstance(token, dict):
            token = token["token"]

        start = text.find(token, search_position)
        if start < 0:
            raise ValueError(
                f"Token alignment failed for {token!r} after character {search_position}"
            )
        end = start + len(token)
        search_position = end

        if tag == "O":
            close_entity()
            continue

        if tag.startswith("B-"):
            close_entity()
            current_entity = {
                "start": start,
                "end": end,
                "label": tag[2:],
            }
        elif tag.startswith("I-"):
            label = tag[2:]
            if current_entity is None or current_entity["label"] != label:
                raise ValueError(
                    f"Invalid BIO sequence: {tag} without matching B-{label}"
                )
            current_entity["end"] = end
        else:
            raise ValueError(f"Unknown BIO tag: {tag}")

    close_entity()
    return entities

# Validate span reconstruction across the complete test split.
gold_entities = [bio_to_entities(sample) for sample in test_data]
assert all(
    0 <= entity["start"] < entity["end"] <= len(sample["text"])
    for sample, entities in zip(test_data, gold_entities)
    for entity in entities
)
print("BIO-to-span conversion verified for", len(test_data), "test samples.")
print("Example gold entities:", gold_entities[0])

BIO-to-span conversion verified for 140 test samples.
Example gold entities: [{'start': 0, 'end': 10, 'label': 'PERSON'}, {'start': 21, 'end': 30, 'label': 'UPI'}]


## 4. Exact-span evaluation utilities

A true positive requires an exact match on `(start, end, label)`. The report includes precision, recall, F1, and gold support for each entity type. Macro F1 is the unweighted mean across the seven project labels.

In [4]:
def evaluate_predictions(results, labels=LABELS):
    rows = []

    for label in labels:
        tp = fp = fn = support = 0

        for sample in results:
            gold = {
                (e["start"], e["end"], e["label"])
                for e in sample["ground_truth"]
                if e["label"] == label
            }
            predicted = {
                (e["start"], e["end"], e["label"])
                for e in sample["predictions"]
                if e["label"] == label
            }

            tp += len(gold & predicted)
            fp += len(predicted - gold)
            fn += len(gold - predicted)
            support += len(gold)

        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        rows.append({
            "Entity": label,
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1,
            "Support": support,
        })

    return pd.DataFrame(rows).set_index("Entity").round(4)


def evaluate_by_language(results, labels=LABELS):
    language_results = {}
    languages = sorted({sample["language"] for sample in results})

    for language in languages:
        subset = [sample for sample in results if sample["language"] == language]
        language_results[language] = evaluate_predictions(subset, labels)

    return language_results


def print_macro_f1(metrics, title):
    print(f"{title} Macro F1:", round(metrics["F1-score"].mean(), 4))

## 5. Run and evaluate the Regex baseline

In [5]:
regex_results = []

for index, sample in enumerate(test_data):
    regex_results.append({
        "sample_id": index,
        "language": sample["language"],
        "text": sample["text"],
        "predictions": regex_detect(sample["text"]),
        "ground_truth": gold_entities[index],
    })

regex_metrics = evaluate_predictions(regex_results)
display(regex_metrics)
print_macro_f1(regex_metrics, "Regex")

regex_language_metrics = evaluate_by_language(regex_results)
for language, metrics in regex_language_metrics.items():
    print(f"\\nRegex – {language}")
    display(metrics)

,Precision,Recall,F1-score,Support
Entity,,,,
PERSON,0.0,0.0,0.0,40
ADDRESS,0.0,0.0,0.0,20
PHONE,1.0,1.0,1.0,40
AADHAAR,1.0,1.0,1.0,40
PAN,1.0,1.0,1.0,20
EMAIL,1.0,1.0,1.0,20
UPI,1.0,1.0,1.0,40


Regex Macro F1: 0.7143
\nRegex – en


,Precision,Recall,F1-score,Support
Entity,,,,
PERSON,0.0,0.0,0.0,20
ADDRESS,0.0,0.0,0.0,0
PHONE,1.0,1.0,1.0,40
AADHAAR,0.0,0.0,0.0,0
PAN,1.0,1.0,1.0,20
EMAIL,1.0,1.0,1.0,20
UPI,0.0,0.0,0.0,0


\nRegex – hi


,Precision,Recall,F1-score,Support
Entity,,,,
PERSON,0.0,0.0,0.0,0
ADDRESS,0.0,0.0,0.0,20
PHONE,0.0,0.0,0.0,0
AADHAAR,1.0,1.0,1.0,40
PAN,0.0,0.0,0.0,0
EMAIL,0.0,0.0,0.0,0
UPI,0.0,0.0,0.0,0


\nRegex – hinglish


,Precision,Recall,F1-score,Support
Entity,,,,
PERSON,0.0,0.0,0.0,20
ADDRESS,0.0,0.0,0.0,0
PHONE,0.0,0.0,0.0,0
AADHAAR,0.0,0.0,0.0,0
PAN,0.0,0.0,0.0,0
EMAIL,0.0,0.0,0.0,0
UPI,1.0,1.0,1.0,40


## 6. Save Regex baseline results

In [6]:
regex_metrics.to_csv(RESULTS_DIR / "regex_baseline_results.csv")

regex_language_rows = []
for language, metrics in regex_language_metrics.items():
    temp = metrics.reset_index()
    temp.insert(0, "Language", language)
    regex_language_rows.append(temp)

pd.concat(regex_language_rows, ignore_index=True).to_csv(
    RESULTS_DIR / "regex_baseline_language_results.csv", index=False
)

with open(RESULTS_DIR / "regex_predictions.json", "w", encoding="utf-8") as f:
    json.dump(regex_results, f, ensure_ascii=False, indent=2)

print("Regex metrics and predictions saved.")

Regex metrics and predictions saved.


## 7. Initialize Microsoft Presidio

Presidio uses the English spaCy model in this experiment. If the model is not installed in the current environment, install `en_core_web_sm` before running this section. The analyzer is intentionally configured for English only.

In [7]:
from presidio_analyzer import AnalyzerEngine, Pattern, PatternRecognizer
from presidio_analyzer.nlp_engine import NlpEngineProvider

configuration = {
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "en", "model_name": "en_core_web_sm"}],
}

provider = NlpEngineProvider(nlp_configuration=configuration)
nlp_engine = provider.create_engine()

analyzer = AnalyzerEngine(
    nlp_engine=nlp_engine,
    supported_languages=["en"],
)

print("Presidio initialized successfully.")

/Users/navyaa09/IndiaPII/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1118 15:22:50.536000 11730 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Presidio initialized successfully.


## 8. Register custom Indian PII recognizers

Custom recognizers are added for Aadhaar, PAN, Indian phone numbers, UPI IDs, and English-formatted addresses. Presidio's built-in recognizers continue to provide its PERSON and email detection.

In [8]:
custom_recognizers = [
    PatternRecognizer(
        supported_entity="AADHAAR",
        patterns=[Pattern(
            name="aadhaar_pattern",
            regex=r"(?<!\d)\d{4}[\s-]?\d{4}[\s-]?\d{4}(?!\d)",
            score=0.85,
        )],
    ),
    PatternRecognizer(
        supported_entity="PAN",
        patterns=[Pattern(
            name="pan_pattern",
            regex=r"(?i)(?<![A-Z])[A-Z]{5}\d{4}[A-Z](?![A-Z])",
            score=0.95,
        )],
    ),
    PatternRecognizer(
        supported_entity="PHONE",
        patterns=[Pattern(
            name="indian_phone",
            regex=r"(?<!\d)(?:\+91[\s-]?)?[6-9]\d{9}(?!\d)",
            score=0.90,
        )],
    ),
    PatternRecognizer(
        supported_entity="UPI",
        patterns=[Pattern(
            name="upi_pattern",
            regex=r"(?i)\b[a-z0-9._-]+@(?:okaxis|ybl|oksbi|paytm|okhdfcbank|ibl)\b",
            score=0.95,
        )],
    ),
    PatternRecognizer(
        supported_entity="ADDRESS",
        patterns=[Pattern(
            name="address_pattern",
            regex=r"(?i)\b(?:flat|house|plot|sector|road|street|layout|nagar|colony)\s+[A-Za-z0-9., -]{2,50}",
            score=0.65,
        )],
    ),
]

for recognizer in custom_recognizers:
    analyzer.registry.add_recognizer(recognizer)

print(f"Registered {len(custom_recognizers)} custom recognizers.")

Registered 5 custom recognizers.


## 9. Run and evaluate the Presidio baseline

The analyzer is called with `language="en"` because that is the only language configured. Hindi and Hinglish samples are included in the overall test-set evaluation, but their scores are out-of-language results for this English Presidio setup.

In [9]:
PRESIDIO_LABEL_MAP = {
    "PERSON": "PERSON",
    "PHONE": "PHONE",
    "PHONE_NUMBER": "PHONE",
    "AADHAAR": "AADHAAR",
    "PAN": "PAN",
    "EMAIL_ADDRESS": "EMAIL",
    "EMAIL": "EMAIL",
    "UPI": "UPI",
    "ADDRESS": "ADDRESS",
}

presidio_results = []

for index, sample in enumerate(test_data):
    text = sample["text"]

    raw_predictions = analyzer.analyze(
        text=text,
        language="en",
    )

    predictions = []
    for result in raw_predictions:
        label = PRESIDIO_LABEL_MAP.get(result.entity_type)
        if label is None:
            continue

        predictions.append({
            "start": result.start,
            "end": result.end,
            "label": label,
            "text": text[result.start:result.end],
            "score": float(result.score),
        })

    presidio_results.append({
        "sample_id": index,
        "language": sample["language"],
        "text": text,
        "predictions": predictions,
        "ground_truth": gold_entities[index],
    })

presidio_metrics = evaluate_predictions(presidio_results)
display(presidio_metrics)
print_macro_f1(presidio_metrics, "Presidio")

presidio_language_metrics = evaluate_by_language(presidio_results)
for language, metrics in presidio_language_metrics.items():
    print(f"\\nPresidio – {language}")
    display(metrics)

,Precision,Recall,F1-score,Support
Entity,,,,
PERSON,0.3415,0.35,0.3457,40
ADDRESS,0.0000,0.00,0.0000,20
PHONE,1.0000,1.00,1.0000,40
AADHAAR,1.0000,1.00,1.0000,40
PAN,1.0000,1.00,1.0000,20
EMAIL,1.0000,1.00,1.0000,20
UPI,1.0000,1.00,1.0000,40


Presidio Macro F1: 0.7637
\nPresidio – en


,Precision,Recall,F1-score,Support
Entity,,,,
PERSON,1.0,0.6,0.75,20
ADDRESS,0.0,0.0,0.00,0
PHONE,1.0,1.0,1.00,40
AADHAAR,0.0,0.0,0.00,0
PAN,1.0,1.0,1.00,20
EMAIL,1.0,1.0,1.00,20
UPI,0.0,0.0,0.00,0


\nPresidio – hi


,Precision,Recall,F1-score,Support
Entity,,,,
PERSON,0.0,0.0,0.0,0
ADDRESS,0.0,0.0,0.0,20
PHONE,0.0,0.0,0.0,0
AADHAAR,1.0,1.0,1.0,40
PAN,0.0,0.0,0.0,0
EMAIL,0.0,0.0,0.0,0
UPI,0.0,0.0,0.0,0


\nPresidio – hinglish


,Precision,Recall,F1-score,Support
Entity,,,,
PERSON,0.0714,0.1,0.0833,20
ADDRESS,0.0000,0.0,0.0000,0
PHONE,0.0000,0.0,0.0000,0
AADHAAR,0.0000,0.0,0.0000,0
PAN,0.0000,0.0,0.0000,0
EMAIL,0.0000,0.0,0.0000,0
UPI,1.0000,1.0,1.0000,40


## 10. Compare the two baselines

In [10]:
baseline_comparison = pd.DataFrame({
    "Regex Precision": regex_metrics["Precision"],
    "Regex Recall": regex_metrics["Recall"],
    "Regex F1": regex_metrics["F1-score"],
    "Presidio Precision": presidio_metrics["Precision"],
    "Presidio Recall": presidio_metrics["Recall"],
    "Presidio F1": presidio_metrics["F1-score"],
    "Gold Support": regex_metrics["Support"],
})

display(baseline_comparison)

macro_comparison = pd.DataFrame({
    "Model": ["Regex", "Presidio"],
    "Macro F1": [
        regex_metrics["F1-score"].mean(),
        presidio_metrics["F1-score"].mean(),
    ],
})
display(macro_comparison)

,Regex Precision,Regex Recall,Regex F1,Presidio Precision,Presidio Recall,Presidio F1,Gold Support
Entity,,,,,,,
PERSON,0.0,0.0,0.0,0.3415,0.35,0.3457,40
ADDRESS,0.0,0.0,0.0,0.0000,0.00,0.0000,20
PHONE,1.0,1.0,1.0,1.0000,1.00,1.0000,40
AADHAAR,1.0,1.0,1.0,1.0000,1.00,1.0000,40
PAN,1.0,1.0,1.0,1.0000,1.00,1.0000,20
EMAIL,1.0,1.0,1.0,1.0000,1.00,1.0000,20
UPI,1.0,1.0,1.0,1.0000,1.00,1.0000,40


,Model,Macro F1
0,Regex,0.714286
1,Presidio,0.763671


## 11. Save all baseline outputs

In [11]:
presidio_metrics.to_csv(RESULTS_DIR / "presidio_baseline_results.csv")
baseline_comparison.to_csv(RESULTS_DIR / "baseline_comparison.csv")
macro_comparison.to_csv(RESULTS_DIR / "baseline_macro_comparison.csv", index=False)

presidio_language_rows = []
for language, metrics in presidio_language_metrics.items():
    temp = metrics.reset_index()
    temp.insert(0, "Language", language)
    presidio_language_rows.append(temp)

pd.concat(presidio_language_rows, ignore_index=True).to_csv(
    RESULTS_DIR / "presidio_baseline_language_results.csv", index=False
)

with open(RESULTS_DIR / "presidio_predictions.json", "w", encoding="utf-8") as f:
    json.dump(presidio_results, f, ensure_ascii=False, indent=2)

print("All baseline metrics and prediction files saved to:", RESULTS_DIR)

All baseline metrics and prediction files saved to: /Users/navyaa09/IndiaPII/results


## 12. Interpretation and limitations

- **Regex** is a format-matching baseline. It is expected to perform better on structured identifiers than on PERSON and free-form ADDRESS entities.
- **Presidio** combines its built-in English NLP recognizers with custom Indian-format recognizers. Its configured language support in this notebook is English only.
- **Hindi/Hinglish scores for Presidio are out-of-language measurements** and should not be presented as native multilingual Presidio performance.
- Exact-span evaluation is strict: a partially detected identifier is counted as an error.
- The dataset is synthetic. These results do not establish performance on real user data.
- Pattern matches do not establish that Aadhaar or PAN values are genuine or valid.

Use these baselines as reference points for the multilingual MuRIL model evaluated in Notebook 03. Do not compare results from different test splits.